<a href="https://colab.research.google.com/github/tavishgrade/nlp-language-model-portfolio/blob/main/Talktorial_3_Semantics_student_TavishGrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talktorial 3 — Semantics: WordNet Similarity & Word Sense Disambiguation

**Graduate NLP · Week 5 · Exposure #2** (lecture → **talktorial** → peer discussion)

In the lecture you saw meaning as **relations** (WordNet) and as **context**. Here you'll walk a
real lexical database, compute the **similarity measures** from class, and disambiguate a word by
its context — all knowledge-based, no embeddings yet (those are Week 6).

> ⚠️ **Colab users — do this first:** **File → Save a copy in Drive.**

**By the end you will be able to:**
1. Explore **synsets** (senses) and relations in WordNet
2. Find the **least common subsumer** that the similarity measures use
3. Compute **path / Wu-Palmer / Resnik / Lin / JCN** similarity
4. Run **Lesk** word sense disambiguation
5. Reason about what "similar" vs "related" really means

**Submit:** answer the prediction questions below, run all cells (outputs saved), complete the
**AI Verification Log**, answer the **reflection questions** at the end, then upload the
`.ipynb` to Canvas by **Sun 11:59 PM**.

## Before you code: make some predictions

Type your answers in this cell (double-click to edit). **One or two sentences each, and every answer
needs a reason.** Predictions are graded for being real, not for being right.

- **Q1.** How many senses do you predict the word **"bass"** has in WordNet — and why that many?
  > *Your answer:* Oh gosh, reasoning for this answer would be the count in my head as I try to think of the various definitions of bass. One for fish, two for instrument, three for low end sound, maybe four for a verb of adding more low end sound, and I'm guessing three more given my presumed lack of definition knowledge for a total of seven.
- **Q2.** Is **"dog"** more similar to **"cat"** or to **"car"**, roughly how big is the gap
  (path similarity is 0–1), and why?
  > *Your answer:* If we're going by the standard similarity equation of 1 over the length of the path connecting the two words, cat and dog are both directly under a category of domesticated mammals (or pets, or houshold animals, etc) and certainly no matter the category only two steps away (per the Professor's example of sedan and sport being path = 2). For dog and car I can't quickly picture what the parent they might share, I'll guess at a path length of 10, resulting in a path similarity of .5 for dog and cat versus .2 for dog and car. A number closer to one indicates more similar, so dog and cat would be more similar.
- **Q3.** Will **Lesk** correctly tell apart *"He played the bass guitar"* from *"He caught a bass"*?
  Why or why not?
  > *Your answer:* I think so yes. Given the example of computer mouse, using the added number of context overlaps, I would assume words like played and guitar would create a higher overlap number for the sense of instrument for bass.

## Part 0 — Setup

Provided. Downloads WordNet + the information-content counts (safe to re-run).

In [ ]:
import json   # used by the artifact cell in Part 7 -- four people hit a NameError here in Talktorial 2
import nltk
for pkg in ["wordnet", "wordnet_ic", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import wordnet as wn, wordnet_ic
from nltk.tokenize import word_tokenize
from nltk.wsd import lesk

print("WordNet ready:", len(list(wn.all_synsets())), "synsets")

WordNet ready: 117659 synsets


## Part 1 — A word has many senses

In [ ]:
# TODO: Get all synsets of "bass" with wn.synsets("bass").
# For each, print its .name(), .pos(), and .definition().
# Then answer the question in the cell below.
n = 0

# create a list of synsets for bass
bass_synsets = wn.synsets("bass")
bass_synset_names = [s.name() for s in wn.synsets("bass")]

# iterate through the synsets and print out name, POS, and definition
for synset in bass_synsets:
  n += 1
  print(n, synset.name(), synset.pos(), synset.definition())

print(bass_synset_names)

1 bass.n.01 n the lowest part of the musical range
2 bass.n.02 n the lowest part in polyphonic music
3 bass.n.03 n an adult male singer with the lowest voice
4 sea_bass.n.01 n the lean flesh of a saltwater fish of the family Serranidae
5 freshwater_bass.n.01 n any of various North American freshwater fish with lean flesh (especially of the genus Micropterus)
6 bass.n.06 n the lowest adult male singing voice
7 bass.n.07 n the member with the lowest range of a family of musical instruments
8 bass.n.08 n nontechnical name for any of numerous edible marine and freshwater spiny-finned fishes
9 bass.s.01 s having or denoting a low vocal or instrumental range
['bass.n.01', 'bass.n.02', 'bass.n.03', 'sea_bass.n.01', 'freshwater_bass.n.01', 'bass.n.06', 'bass.n.07', 'bass.n.08', 'bass.s.01']


**Does the count match your Q1 prediction?** If not, what did you miss — parts of speech, or senses you had not thought of?
> *Your answer:* I got pretty close, it was certainly good that I added a few more on top of the ones I could think of. I did not get them all, and I definitely didn't consider the various types of bass fish. I'm sure if I had that lived knowledge I would have. On top of that, I added one as a verb for freqency level that is not listed, and maybe I leaned into personal slang more for that guess.

In [ ]:
# TODO: For wn.synset("dog.n.01"), print its definition, .lemma_names() (synonyms),
# .examples(), .hypernyms() (what it IS-A), and a few .hyponyms() (kinds of it).
s = wn.synset("dog.n.01")

print("Definition:", s.definition())
print("Lemma Names:", s.lemma_names())
print("Examples:", s.examples())
print("Hyernyms:", s.hypernyms())
print("Hyponyms (first 8):", s.hyponyms()[:8])

Definition: a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds
Lemma Names: ['dog', 'domestic_dog', 'Canis_familiaris']
Examples: ['the dog barked all night']
Hyernyms: [Synset('domestic_animal.n.01'), Synset('canine.n.02')]
Hyponyms (first 8): [Synset('basenji.n.01'), Synset('great_pyrenees.n.01'), Synset('working_dog.n.01'), Synset('poodle.n.01'), Synset('toy_dog.n.01'), Synset('mexican_hairless.n.01'), Synset('puppy.n.01'), Synset('hunting_dog.n.01')]


## Part 2 — The least common subsumer (LCS)
The similarity measures all hinge on the **deepest shared ancestor** of two concepts.

In [ ]:
# TODO: Use dog.lowest_common_hypernyms(cat) to find the least common subsumer of
# dog.n.01 and cat.n.01. Print it and its definition. (This is the node Wu-Palmer,
# Resnik, etc. all use.)
dog, cat = wn.synset("dog.n.01"), wn.synset("cat.n.01")

lcs = dog.lowest_common_hypernyms(cat)
print(lcs)
print(lcs[0].definition())

[Synset('carnivore.n.01')]
a terrestrial or aquatic flesh-eating mammal


## Part 3 — Path-based similarity

In [ ]:
# TODO: Compute path_similarity and wup_similarity between dog.n.01 and BOTH
# cat.n.01 and car.n.01. Then answer the question in the cell below.
car = wn.synset("car.n.01")

# for information purposes, display the lowest common hypernyms of both parirings
lcs_dog_cat = dog.lowest_common_hypernyms(cat)
lcs_dog_car = dog.lowest_common_hypernyms(car)
print(lcs_dog_cat)
print(lcs_dog_car)
print()

# calculate the path similarity of both pairings
print("dog/cat path:", dog.path_similarity(cat))
print("dog/car path:", dog.path_similarity(car))
print()

# calculate the Wu and Palmer similarities of both pairings
print("dog/cat wup:", dog.wup_similarity(cat))
print("dog/car wup:", dog.wup_similarity(car))

[Synset('carnivore.n.01')]
[Synset('whole.n.02')]

dog/cat path: 0.2
dog/car path: 0.07692307692307693

dog/cat wup: 0.8571428571428571
dog/car wup: 0.4


**Which pair is more similar, and does your Q2 prediction hold** — both the direction and the size of the gap?
> *Your answer:* I was correct in the most similar versus least similar, but I didn't correctly gauge how detailed the categories are in WordNet, or even the specific pathways they might take. I personally like the alterations Wu and Palmer made to the similarity equation taking depth of LCS into account. It more accurately reflects the relations that dog and cat do have with each other on the scale of 0-1, even though the ratios of these two results are similar to each other.

## Part 4 — Information-content similarity (Resnik, Lin, JCN)
These weight the shared ancestor by how *specific* (rare) it is — `IC(c) = -log P(c)`.

In [ ]:
# TODO: Load brown_ic = wordnet_ic.ic("ic-brown.dat"). Then for dog vs cat and
# dog vs car, print res_similarity, lin_similarity, jcn_similarity (each takes the
# other synset AND brown_ic). Then answer the question in the cell below.
brown_ic = wordnet_ic.ic("ic-brown.dat")

# print the resnik similarity results of dog with cat and car
print("dog/cat res:", round(dog.res_similarity(cat, brown_ic),2))
print("dog/car res:", round(dog.res_similarity(car, brown_ic),2))
print()

# print the lin similarity results of dog with cat and car
print("dog/cat lin:", round(dog.lin_similarity(cat, brown_ic),2))
print("dog/car lin:", round(dog.lin_similarity(car, brown_ic),2))
print()

# print the jiang-conrath similarity results of dog with cat and car
print("dog/cat jcn:", round(dog.jcn_similarity(cat, brown_ic),2))
print("dog/car jcn:", round(dog.jcn_similarity(car, brown_ic),2))
print()

# calculate the ratios between the dog-cat and dog-car similarities for all findings
print("ratio of path-sim: ", round(dog.path_similarity(cat)/dog.path_similarity(car),2))
print("ratio of wup-sim: ", round(dog.wup_similarity(cat)/dog.wup_similarity(car),2))
print("ratio of res-sim: ", round(dog.res_similarity(cat, brown_ic)/dog.res_similarity(car, brown_ic)))
print("ratio of lin-sim: ", round(dog.lin_similarity(cat, brown_ic)/dog.lin_similarity(car, brown_ic)))
print("ratio of jcn-sim: ", round(dog.jcn_similarity(cat, brown_ic)/dog.jcn_similarity(car, brown_ic)))

dog/cat res: 7.91
dog/car res: 1.53

dog/cat lin: 0.88
dog/car lin: 0.18

dog/cat jcn: 0.45
dog/car jcn: 0.07

ratio of path-sim:  2.6
ratio of wup-sim:  2.14
ratio of res-sim:  5
ratio of lin-sim:  5
ratio of jcn-sim:  6


**How do these compare to the path-based numbers?** Same ordering? Same size of gap? (Note that JCN and Resnik are not on a 0–1 scale.)
> *Your answer:* Similar to the path-based numbers, the ordering is the same for all calculations, with the dog-cat similarity deemed higher than dog-car. Doing the math on the gap in the above code section, I see that the first two similarity equations that didn't involve the depth/specificity of have a much smaller gap better these two word pairings. Whereas, the following three variations using the information content show a much larger gap between these pairings.

### ✅ Verification checkpoint #1
Ask your AI assistant: *"What is the difference between Resnik and Lin similarity?"* Verify against
the **Semantics lecture** (the IC-measures slide), the **NLTK WordNet docs**, and
**SLP3 "Word Senses and WordNet"**. Record it in Part 6.

## Part 5 — Word sense disambiguation with Lesk

In [ ]:
# TODO: For each sentence below, tokenize it and call lesk(tokens, "bass").
# Print the chosen synset name + definition. Then answer the questions in the cell below.
for sent in ["He played the bass guitar in the band.",
             "He caught a huge bass while fishing."]:
             tokens = word_tokenize(sent)
             sense = lesk(tokens, "bass")
             print("Sentence:", sent)
             print("Chosen synset of bass:", sense.name());
             print("Definition:", sense.definition())
             print()

Sentence: He played the bass guitar in the band.
Chosen synset of bass: bass.n.02
Definition: the lowest part in polyphonic music

Sentence: He caught a huge bass while fishing.
Chosen synset of bass: sea_bass.n.01
Definition: the lean flesh of a saltwater fish of the family Serranidae



**Did it run?** Did Lesk pick the sense you expected for **each** sentence? (Your Q3.)
> *Your answer:* Definitely not! I thought based on the lecture that it would work as common sense would allow. I didn't anticipate how overlapping these senses would be in terms of connected words. When I made my guess, I still had only one type of fish, and I figured guitar and played would overrule bass as a level of sound frequency.

**If not — is that a bug in your code, or is that the result?** Look at the definition Lesk chose: which of its words overlap the sentence?
> *Your answer:* It is not a bug in my code. When looking at the definition for the bass guitar, I don't see any non-stop words that overlap, so I assume the higher score came from something I don't see in this definition? I looked back at all 9 definitions from the first question and I still don't understand that one. For the fish sentence, the overlapping words in the definition and the sentence are fish, but why would it lead to this specific fish? I have more questions than answers.

### ✅ Verification checkpoint #2
Did Lesk get both senses right? Ask your AI assistant *why* Lesk can fail, and verify against the
Semantics lecture (the WSD section) and **SLP3 "Word Senses and WordNet"** (§WSD). Record it in Part 6.

## Part 6 — AI Verification Log

One block per checkpoint: the **question you asked**, the **claim** that came back, the **source** you
checked it against, your **verdict**, and **what you found when you checked** — what the source actually
said, or what your own output actually showed. Then add at least one claim you chose yourself.

**How the log is graded** — the same four rows as the assignment page:
- **The question.** A real question, and a follow-up where the first answer was thin. *"Isn't Lin just
  Resnik normalised?"* is fishing for agreement; *"What does Lin similarity compute?"* is a question.
- **The source.** The right kind for the kind of claim: the textbook or lecture for a definition, the
  NLTK docs for how a function behaves, **your own output** for a claim about what the code does.
- **Thoroughness.** Every checkpoint, plus a claim you chose yourself.
- **What you found.** The finding says what the source *said* or what your output *showed*. Naming a
  caveat, a boundary condition, or a case where the claim breaks is what makes a block excellent
  rather than satisfactory. The AI does not have to be wrong.

*(Double-click this cell to edit. Type after each bold label, then Shift+Enter to render it.)*

**A worked example, filled in so you can see the bar.** This one is not yours to answer:

- **What you asked:** "Is WordNet's `path_similarity` symmetric — is path_similarity(a,b) the same as (b,a)?"
- **What it told you:** Yes, it is symmetric.
- **Source checked:** NLTK WordNet docs, plus my own output
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** Symmetric, yes — but it returns `None` when no path exists, which
  happens across parts of speech. So the two calls agree, and sometimes they agree on nothing. The AI
  did not mention `None`.

**The same block, written the way that does not count.** Same question, same source:

- **What you found when you checked:** The docs confirm that path_similarity is symmetric. ✅

That repeats the claim. It does not say what the docs said, and it did not run the two calls. A
finding that could have been written without opening the source is not a check.

---

### Claim 1 — checkpoint #1
- **What you asked:** What is the difference between Resnik and Lin similarity?
- **What it told you:** It gave me the equations for both Resnik and Lim similarity. It briefed me first on what IC is and what LCS is and denoted that it's typically used with WordNet-style taxonomies.
- **Source checked:** NLTK doc, Lecture slides and video
- **Verdict:** ⚠️ needs caveat
- **What you found when you checked:** For noting the differences, the equations clearly showed the difference exactly as the lecture and NLTK documents display them.  When checking the NLTK documents, they go further and list a specific caveat not discussed by the AI that the result is "dependent on the corpus used" to create the IC value. This caveat is vital when putting a comparison of the two against each other.

### Claim 2 — checkpoint #2
- **What you asked:** Why can Lesk fail?
- **What it told you:** The AI gave a long list of reasons why Lesk can fail, and actually answered a lot of the questions I listed in my response to the Lesk question from earlier. The final sentence after some great points was "literal lexical overlap is a weak proxy for semantic similarity." The key finding takeaways for me were, short definitions, context and definition may be using synonyms and don't get counted, different senses can be very similar to each other and can then be difficult to separately gauge.
- **Source checked:** SLP3 and Lecture slides and video
- **Verdict:** ⚠️ needs caveat
- **What you found when you checked:** To be fair, the AI went into details not discussed in the Book or the Lecture, but there are always caveats of course. The one note that was missing from the AI explanation but was included in both the lecture and book was that Lesk doesn't used any supervised data so the lack of context seeps in.

### Claim 3 — one you chose yourself
*A claim about what the code does is checked against your own output — your Lesk results from Part 5,
or a similarity number from Part 3 or 4, belong here.*
- **What you asked:** Based on Lesk algorithms, which definition of "bass" will be chosen for the sentences in Part 5?
- **What it told you:** For "He played the bass guitar in the band.", Lesk returned bass.n.07 n the member with the lowest range of a family of musical instruments.
"For He caught a huge bass while fishing.", Lesk returned bass.n.08 n nontechnical name for any of numerous edible marine and freshwater spiny-finned fishes.
- **Source checked:** My own output Part 5
- **Verdict:** ❌ wrong
- **What you found when you checked:** Similar to the ambiguous problem of the garden path problem from the prior talktorial, the AI LLM became tripped up over this type of question. In this example, the AI clearly chose the answers that make the most contexual sense, which makes sense for the AI which uses context. However, the answers I got while using the Lesk algorith did not lean towards the general definitions when stuck between options and went with def 2 and 4. The results do not line up.

**Verdict key:** ✅ correct · ⚠️ partly right / needs caveat · ❌ wrong.

## Part 7 — Save your artifact (shared format → final project)

In [ ]:
# TODO: Build a dict `artifact` in the shared envelope:
#   {"week": 5, "task": "semantic-similarity", "source": "WordNet (NLTK)", "outputs": {...}}
# `outputs` is a DICT with named keys. Fill it from the variables and calls in this
# notebook -- the list of bass synset names, and two or three of your similarity
# numbers -- not by typing values in. Save it with json.dump as
# talktorial3_artifact.json, then print it.
artifact = {
    "week": 5,
    "task": "semantic-similarity",
    "source": "WordNet (NLTK)",
    "outputs": {
        "bass_synset_names": bass_synset_names,
        "res_similarity": {
            "dog/cat": round(dog.res_similarity(cat, brown_ic),2),
            "dog/car": round(dog.res_similarity(car, brown_ic),2)
        },
        "lin_similarity": {
            "dog/cat": round(dog.lin_similarity(cat, brown_ic),2),
            "dog/car": round(dog.lin_similarity(car, brown_ic),2)
        },
        "jcn_similarity": {
            "dog/cat": round(dog.jcn_similarity(cat, brown_ic),2),
            "dog/car": round(dog.jcn_similarity(car, brown_ic),2)
        }
    }
}

# saving the doc
with open("talktorial3_artifact.json", "w") as f:
  json.dump(artifact, f, indent=2)

print(json.dumps(artifact, indent=2))

{
  "week": 5,
  "task": "semantic-similarity",
  "source": "WordNet (NLTK)",
  "outputs": {
    "bass_synset_names": [
      "bass.n.01",
      "bass.n.02",
      "bass.n.03",
      "sea_bass.n.01",
      "freshwater_bass.n.01",
      "bass.n.06",
      "bass.n.07",
      "bass.n.08",
      "bass.s.01"
    ],
    "res_similarity": {
      "dog/cat": 7.91,
      "dog/car": 1.53
    },
    "lin_similarity": {
      "dog/cat": 0.88,
      "dog/car": 0.18
    },
    "jcn_similarity": {
      "dog/cat": 0.45,
      "dog/car": 0.07
    }
  }
}


In [ ]:
# Provided: checks the artifact is in the shared shape. If it fails, fix the cell above.
import json, os
assert os.path.exists("talktorial3_artifact.json"), "talktorial3_artifact.json was not written"
_a = json.load(open("talktorial3_artifact.json"))
_missing = {"week", "task", "source", "outputs"} - set(_a)
assert not _missing, f"envelope is missing {_missing}"
assert isinstance(_a["outputs"], dict) and _a["outputs"], "outputs must be a non-empty dict with named keys"
print("artifact OK — outputs keys:", sorted(_a["outputs"]))

artifact OK — outputs keys: ['bass_synset_names', 'jcn_similarity', 'lin_similarity', 'res_similarity']


## Before you submit — a few questions, honestly answered

These are not graded for being impressive. They are graded for being real, and they are
how I find out what is not landing while there is still time to do something about it.
*(Double-click this cell to type your answers, then Shift+Enter.)*

**1. What surprised you?**
*(Scroll back to your predictions at the top — where was your intuition wrong, and why?)*

> *Your answer:* Absolutely the question about Lesk and the result surprised me. Listening back to the lecture I hear you say you made it work, so clearly I was convinced by that one great example that it was a good algorithm, but I mistakenly forgot how specialized comparing words to words can get without context. I would assume guitar and played would have had an effect on bass, but none of the definitions included either of those words, making those words in the sentence ineffective it seems.

**2. What got you stuck this week, and what did you do about it?**
*("I am still stuck on X" is a complete and fully-credited answer.)*

> *Your answer:* It's definitely shown in my response to the Lesk results, I got really stuck wondering why the chosen definitions were so off. I listed of several questions that kept me thinking. Then on the next AI check, I simply wrote the question I was told to, and the results given back with the chatty AI bot actually really helped to expand my understanding of how the Lesk algorithm works.

**3. What is one question you still have?**

> *Your answer:* I think my questions will be answered in time, which is how do these methods and algorithms relate to what we see now in technology?

---

### Working with AI

*There is no right answer here — "it frustrated me" and "I am not sure it helped" are as useful to me as anything positive. Answer honestly rather than helpfully.*

**a. Which AI did you use this week?** Name the assistant, and the model if you know it — e.g. "ChatGPT", "Claude", "Gemini", "Copilot", a local model, or "none this week", which is a perfectly good answer.

> *Your answer:* Copilot for clarifying, ChatGPT for the AI verification log

**b. After completing this week’s AI-supported task and checking the AI’s response, what did you think and feel about your experience with AI?**

*You might reflect on things such as what surprised you, what confused or frustrated you, what helped you learn, what made you feel more or less confident, or whether checking the AI’s response changed your thinking or feelings.*

> *Your answer:* I think learning more about the systems in general, how the algorithms we're using are not vector based like you say the LLMs are (I think? I will learn more in time), it's a lot easier for me to see why there are differences in the responses. The AI will not know how Lesk results will be because the systems think very differently. I still feel like there's a lot more to understand though.



## Wrap-up

You walked a real lexical database, computed the similarity measures from class (the same ones used
in biomedical NLP over UMLS), and disambiguated a word by its **context**. Notice the theme: meaning
came from **relations** here — next week it comes from **learned vectors**, and the two converge.

**Optional extensions:** try similarity on a pair from your own field; compare Lesk to just picking
the most frequent sense (`wn.synsets(word)[0]`); explore `wn.synset(...).hypernym_paths()`.

**Submit:** Run all cells, then **File → Download .ipynb** → upload to Canvas with the Verification
Log completed.

In [ ]:
# using verbiage from my own field of sound design, I will find the similarity measurements
# find the definition related to my goal of audio
movie_synsets = wn.synsets("movie")
audio_synsets = wn.synsets("audio")
microphone_synsets = wn.synsets("microphone")

# in a function, iterate through the synsets and print out name, POS, and definition
def synset_info(synsets):
  for synset in synsets:
    print(synset.name(), synset.pos(), synset.definition())

synset_info(movie_synsets)
print()
synset_info(audio_synsets)
print()
synset_info(microphone_synsets)

# the selected definitions will be movie.n.01, sound_recording.n.01, and microphone.n.01

movie.n.01 n a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement

audio.n.01 n the audible part of a transmitted signal
audio.n.02 n an audible acoustic wave frequency
sound_recording.n.01 n a recording of acoustic signals
audio.n.04 n the sound elements of television

microphone.n.01 n device for converting sound waves into electrical energy


In [ ]:
# instantiate the chosen definition synset variables
movie, audio, microphone  = wn.synset("movie.n.01"), wn.synset("sound_recording.n.01"), wn.synset("microphone.n.01")

# calculate the lcs for each pairing
lcs_movie_audio = movie.lowest_common_hypernyms(audio)
lcs_movie_microphone = movie.lowest_common_hypernyms(microphone)
lcs_audio_microphone = audio.lowest_common_hypernyms(microphone)

# print out the results
print("Movie and Audio LCS")
print(lcs_movie_audio)
if lcs_movie_audio:
    print(lcs_movie_audio[0].definition())
else:
    print("No common hypernym found.")
print()

print("Movie and Microphone LCS")
print(lcs_movie_microphone)
if lcs_movie_microphone:
    print(lcs_movie_microphone[0].definition())
else:
    print("No common hypernym found.")
print()

print("Audio and Microphone LCS")
print(lcs_audio_microphone)
if lcs_audio_microphone:
    print(lcs_audio_microphone[0].definition())
else:
    print("No common hypernym found.")

Movie and Audio LCS
[Synset('artifact.n.01')]
a man-made object taken as a whole

Movie and Microphone LCS
[Synset('artifact.n.01')]
a man-made object taken as a whole

Audio and Microphone LCS
[Synset('device.n.01')]
an instrumentality invented for a particular purpose


In [ ]:
# print the path similarity of each pairing
print("movie/audio path:", movie.path_similarity(audio))
print("movie/microphone path:", movie.path_similarity(microphone))
print("audio/microphone path:", audio.path_similarity(microphone))
print()

# print the Wu and Palmer similarities of each pairing
print("movie/audio wup:", movie.wup_similarity(audio))
print("movie/microphone wup:", movie.wup_similarity(microphone))
print("audio/microphone wup:", audio.wup_similarity(microphone))
print()

brown_ic = wordnet_ic.ic("ic-brown.dat")

# print the resnik similarity results of each pairing
print("movie/audio res:", round(movie.res_similarity(audio, brown_ic),2))
print("movie/microphone res:", round(movie.res_similarity(microphone, brown_ic),2))
print("audio/microphone res:", round(audio.res_similarity(microphone, brown_ic),2))
print()

# print the lin similarity results of each pairing
print("movie/audio lin:", round(movie.lin_similarity(audio, brown_ic),2))
print("movie/microphone lin:", round(movie.lin_similarity(microphone, brown_ic),2))
print("audio/microphone lin:", round(audio.lin_similarity(microphone, brown_ic),2))
print()

# print the jiang-conrath similarity results of each pairing
print("movie/audio jcn:", round(movie.jcn_similarity(audio, brown_ic),2))
print("movie/microphone jcn:", round(movie.jcn_similarity(microphone, brown_ic),2))
print("audio/microphone jcn:", round(audio.jcn_similarity(microphone, brown_ic),2))
print()

movie/audio path: 0.1111111111111111
movie/microphone path: 0.1
audio/microphone path: 0.125

movie/audio wup: 0.5555555555555556
movie/microphone wup: 0.5263157894736842
audio/microphone wup: 0.6666666666666666

movie/audio res: 2.31
movie/microphone res: 2.31
audio/microphone res: 4.08

movie/audio lin: 0.28
movie/microphone lin: 0.26
audio/microphone lin: 0.44

movie/audio jcn: 0.08
movie/microphone jcn: 0.08
audio/microphone jcn: 0.1

ratio of path-sim:  1.11
ratio of wup-sim:  1.06
ratio of res-sim:  1


All in all, not great.

I could probably pick more exciting pairings next time, but interesting to go down the rabbit hole.